<a href="https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter07/07_01_shock_tube_pressure_solver.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Shock-Tube Pressure and State Solver

**Book:** *Introduction to Compressible Flows*<br>
**Chapter 7:** Unsteady Wave Motion<br>
**Concepts:** shock tube · incident-shock Mach number · contact-surface matching · expansion region

## Learning objectives

- Solve the driver/driven pressure matching equation.
- Recover incident-shock speed and the states behind the shock and expansion.
- Check velocity and pressure compatibility across the contact surface.

## How to use this notebook

Edit the driver/driven pressures and temperatures near the top of the code cell, then run it.

> **Model scope.** The example assumes equal perfect gases on both sides, constant specific heats, and one-dimensional inviscid motion.


In [ ]:
import numpy as np
from scipy.optimize import root_scalar
gamma1 = 1.4
gamma4 = 1.4
R = 287
T1 = 25 + 273.15  # K
T4 = 20 + 273.15  # K
a1 = np.sqrt(gamma1 * R * T1)
a4 = np.sqrt(gamma4 * R * T4)
p4_p1 = 200e3 / 5e3  # 40
def f(p2_p1):
    if p2_p1 <= 1:
        return 1e6
    num = (gamma4 - 1) * (a1 / a4) * (p2_p1 - 1)
    denom = np.sqrt(2 * gamma1 * (2 * gamma1 + (gamma1 + 1) * (p2_p1 - 1)))
    inner = 1 - num / denom
    if inner <= 0:
        return 1e6
    rhs = p2_p1 * (inner ** (-2 * gamma4 / (gamma4 - 1)))
    return rhs - p4_p1
sol = root_scalar(f, method='brentq', bracket=[1.01, 10], xtol=1e-8)
if sol.converged:
    p2_p1 = sol.root
    print(f" Converged: p2/p1 = {p2_p1:.8f}")
    Ms = np.sqrt(((gamma1 + 1) / (2 * gamma1)) * (p2_p1 - 1) + 1)
    Cs= Ms*a1
    print(f" Shock Mach number Ms = {Ms:.4f}")
    T2_T1 = ((2 * gamma1 * Ms**2 - (gamma1 - 1)) * ((gamma1- 1) * Ms**2 + 2)) / ((gamma1 + 1)**2 * Ms**2)
    rho2_rho1 = ((gamma1 + 1) * Ms**2) / ((gamma1- 1) * Ms**2 + 2)
    a2 = a1 * np.sqrt(T2_T1)
    u2 = Cs * (1 - 1 / rho2_rho1)
    u3 = u2
    p3_p4 = (1 - ((gamma4 - 1) / 2) * (u3/a4)) ** (2 * gamma4 / (gamma4 - 1))
    T3_T4 = p3_p4 ** ((gamma4 - 1) / gamma4)
    rho3_rho4 = (T3_T4) ** (1 / (gamma4 - 1))
    print(f" Temperature ratio T2/T1 = {T2_T1:.4f}")
    print(f" Density ratio rho2/rho1 = {rho2_rho1:.4f}")
    print(f" Flow velocity u2 = {u2:.2f} m/s")
    print(f" Temperature ratio T3/T4 = {T3_T4:.4f}")
    print(f" Density ratio rho3/rho4 = {rho3_rho4:.4f}")
    print(f" Pressure ratio p3/p4 = {p3_p4:.4f}")
else:
    print(" Did not converge.")


## Interpretation and verification checklist

- The solved p2/p1 must remain inside the root bracket; widen the bracket only with a physical justification.
- Regions 2 and 3 share velocity and pressure across the contact surface.
- Pressure and density ratios are dimensionless; wave and particle velocities use m/s.

## Reproducibility note

Restart the runtime and run the notebook from top to bottom after changing inputs. Record the input values, units, heat-capacity ratio, numerical tolerances, and dependency versions with any result used in coursework, research, or a publication.

Questions or reproducibility reports are welcome through the repository's issue templates.
